# 법령 PDF (여러 개) → Neo4j 적재 파이프라인

여러 개의 법령 PDF(법률/시행령/시행규칙 등)를 읽어서:

1. 법령 메타데이터 추출 (법령명, 법령종류, 공포일, 시행일) — GPT로 첫 페이지에서 추출
2. 조문 단위로 파싱 (제N조, 제N조의M 패턴 기준으로 분리)
3. 조번호 정규화 (`original_id_normalized`) — 매칭 실패 방지용
4. 조문 본문 안에서 다른 조문을 인용하는 부분 탐지 → 인용관계(REFERENCES) 추출
5. 조문 본문 임베딩 (OpenAI Embeddings)
6. Neo4j에 LAW / ARTICLE 노드 + CONTAINS / REFERENCES 관계로 적재
7. 벡터 인덱스(`article_vector_index`) 생성

**스키마**
- `(:LAW {id, name, law_type, promulgation_date, effective_date})`
- `(:ARTICLE {id, law_id, name, description, original_id, original_id_normalized, source_pdf, embedding})`
- `(:LAW)-[:CONTAINS]->(:ARTICLE)`
- `(:ARTICLE)-[:REFERENCES]->(:ARTICLE)`

> **필요한 것**: `OPENAI_API_KEY`, `NEO4J_URI`/`NEO4J_USERNAME`/`NEO4J_PASSWORD` 환경변수, 법령 PDF들이 들어있는 폴더


## 0. 패키지 설치

In [ ]:
!pip install -q pymupdf openai neo4j python-dotenv tqdm


## 1. 설정

In [31]:
import os
import re
import json
import glob
from pathlib import Path

import fitz  # PyMuPDF
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

# ----- 설정값: 필요에 맞게 수정하세요 -----
PDF_DIR = "data/"   # 법령 PDF들이 들어있는 폴더 (여러 개)

CHAT_MODEL = "gpt-4o"
EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 3072

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "lawdb")

openai_client = OpenAI()
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

pdf_paths = sorted(glob.glob(str(Path(PDF_DIR) / "*.pdf")))
print(f"발견된 PDF {len(pdf_paths)}개")
for p in pdf_paths:
    print(" -", p)


발견된 PDF 16개
 - data\공무원보수규정(대통령령)(제36501호)(20260707).pdf
 - data\공무원수당 등에 관한 규정(대통령령)(제36015호)(20260102).pdf
 - data\군인 재해보상법(법률)(제20640호)(20250708).pdf
 - data\군인 징계령 시행규칙(국방부령)(제01203호)(20260128).pdf
 - data\군인보수법(법률)(제20802호)(20260319).pdf
 - data\군인복지기본법(법률)(제20638호)(20250708).pdf
 - data\군인사법 시행령(대통령령)(제36067호)(20260203).pdf
 - data\군인사법(법률)(제21319호)(20260203).pdf
 - data\군인연금법(법률)(제21065호)(20251001).pdf
 - data\군인의 지위 및 복무에 관한 기본법 시행령(대통령령)(제35925호)(20260108).pdf
 - data\군인의 지위 및 복무에 관한 기본법(법률)(제20641호)(20260108).pdf
 - data\군형법(법률)(제18465호)(20220701).pdf
 - data\병역법 시행령(대통령령)(제35948호)(20260102).pdf
 - data\병역법(법률)(제21769호)(20260609).pdf
 - data\부대관리훈령(국방부훈령)(제3148호)(20260319).pdf
 - data\제대군인지원에 관한 법률(법률)(제19525호)(20240112).pdf


## 2. PDF 원문 텍스트 추출

법령 PDF는 보통 표/이미지보다 텍스트 위주라서, 페이지 전체 텍스트를 순서대로 이어붙이는 방식이 조문 파싱에 유리합니다.
(페이지 머리말/꼬리말이 섞여 들어올 수 있는데, 조문 정규식 파싱 단계에서 대부분 자연히 걸러집니다.)

In [32]:
def extract_full_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages_text = [page.get_text("text") for page in doc]
    doc.close()
    return "\n".join(pages_text)


## 3. 법령 메타데이터 추출 (GPT)

파일마다 처음 1~2페이지 정도의 텍스트를 GPT에 보여주고, 법령명/법령종류/공포일/시행일을 JSON으로 뽑습니다.

In [33]:
def extract_law_metadata(pdf_path: str, full_text: str) -> dict:
    excerpt = full_text[:3000]  # 법령명, 공포/시행일은 보통 앞부분에 있음

    system_prompt = """당신은 법령 문서에서 메타데이터를 추출하는 전문가입니다.
주어진 텍스트에서 아래 필드를 JSON으로 추출하세요. 모르면 빈 문자열로 두세요.

{
  "name": "법령명 (예: 군인사법)",
  "law_type": "법률 | 대통령령 | 총리령 | 부령 | 훈령 | 예규 | 고시 중 하나",
  "promulgation_date": "공포일 (YYYY-MM-DD 형식, 모르면 빈 문자열)",
  "effective_date": "시행일 (YYYY-MM-DD 형식, 모르면 빈 문자열)"
}

오직 JSON만 출력하세요. 설명이나 코드블록 표시 없이 순수 JSON 텍스트만 출력하세요."""

    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": excerpt},
        ],
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r'^```(json)?\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()

    try:
        meta = json.loads(raw)
    except json.JSONDecodeError:
        meta = {"name": Path(pdf_path).stem, "law_type": "", "promulgation_date": "", "effective_date": ""}

    if not meta.get("name"):
        meta["name"] = Path(pdf_path).stem

    # 공백/개행 등으로 인한 문자열 불일치(law_id 매칭 실패)를 막기 위해 모든 필드를 트리밍
    for key in ("name", "law_type", "promulgation_date", "effective_date"):
        if isinstance(meta.get(key), str):
            meta[key] = meta[key].strip()

    return meta


## 4. 조문 파싱 + 조번호 정규화

`제31조`, `제5조의2` 같은 패턴을 기준으로 본문을 조문 단위로 잘라냅니다.
`original_id_normalized`는 공백/한글 서식 없이 `"31"`, `"5-2"` 같은 통일된 형태로 만들어서, 나중에 검색/인용관계 매칭이 안정적으로 되게 합니다.

In [34]:
# 실제 조문 헤더는 거의 항상 "줄 맨 앞"에서 시작합니다 (예: "제31조(직권면직)").
# 본문 안에 등장하는 인용구(예: "「장애인복지법」제2조제2항에 따른")는 문장 중간에 있으므로
# (?:^|\n) 로 "줄 시작 위치"인 경우만 진짜 조문 헤더로 인정합니다.
# 이렇게 안 하면 본문 안의 인용이 새 조문 시작으로 오인되어 조문 경계가 깨집니다.
ARTICLE_PATTERN = re.compile(
    r'(?:^|\n)\s*제(\d+)조(?:의(\d+))?\s*(\([^)]{1,40}\))?',
    re.MULTILINE,
)


def normalize_article_id(num: str, sub: str = None) -> str:
    return f"{num}-{sub}" if sub else num


def build_original_id(num: str, sub: str = None) -> str:
    return f"제{num}조의{sub}" if sub else f"제{num}조"


def parse_articles(full_text: str) -> list[dict]:
    """본문 텍스트를 조문 단위로 분리. 각 조문은 dict(original_id, original_id_normalized, name, content)"""
    matches = list(ARTICLE_PATTERN.finditer(full_text))
    articles = []

    for i, m in enumerate(matches):
        num, sub, title_paren = m.group(1), m.group(2), m.group(3)

        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        content = full_text[start:end].strip()

        # 너무 짧은 내용(목차의 조문 나열, 오탐 등)은 제외
        if len(content) < 10:
            continue

        title = title_paren.strip("()") if title_paren else ""

        articles.append({
            "original_id": build_original_id(num, sub),
            "original_id_normalized": normalize_article_id(num, sub),
            "name": title,
            "content": content,
        })

    return articles


## 5. 조문 인용관계 추출

조문 본문 안에 다른 조문 번호(`제OO조`, `제OO조의M`)가 언급되면 인용관계로 간주합니다.
같은 법령 안의 인용은 `law_id`가 자기 자신, 다른 법령을 명시적으로 인용하는 경우(`「OO법」 제OO조`)는 대상 법령명을 같이 기록해둡니다.

인용 대상 조문이 실제로 존재하는지는 이후 Neo4j 적재 단계에서 `MATCH`로 확인하고, 없으면 관계를 만들지 않습니다 (조용히 스킵).

In [35]:
CROSS_LAW_REF_PATTERN = re.compile(r'「([^」]{1,30})」\s*제(\d+)조(?:의(\d+))?')
SAME_LAW_REF_PATTERN = re.compile(r'제(\d+)조(?:의(\d+))?')


def extract_references(content: str, self_original_id: str, self_law_name: str) -> list[dict]:
    """조문 본문에서 인용된 다른 조문들을 추출. 각 항목은 dict(law_name, original_id_normalized)"""
    refs = []
    seen = set()
    self_law_name = self_law_name.strip()

    # 1) 다른 법령을 명시적으로 인용하는 경우: 「군인사법」 제7조
    for m in CROSS_LAW_REF_PATTERN.finditer(content):
        law_name, num, sub = m.group(1), m.group(2), m.group(3)
        law_name = law_name.strip()  # PDF 추출 시 붙는 trailing space 등을 제거 (매칭 실패 방지)
        norm_id = normalize_article_id(num, sub)
        key = (law_name, norm_id)
        if key in seen:
            continue
        seen.add(key)
        refs.append({"law_name": law_name, "original_id_normalized": norm_id})

    # 2) 법령명 없이 조번호만 언급 (같은 법령 내 인용으로 간주)
    #    단, 위에서 이미 명시적으로 잡힌 「법령」제N조 뒤의 제N조 부분은 제외하기 위해
    #    cross-law 패턴이 매치된 구간을 마스킹한 뒤 탐색
    masked = CROSS_LAW_REF_PATTERN.sub(lambda mm: " " * len(mm.group(0)), content)

    for m in SAME_LAW_REF_PATTERN.finditer(masked):
        num, sub = m.group(1), m.group(2)
        norm_id = normalize_article_id(num, sub)
        if norm_id == self_original_id:
            continue  # 자기 자신 인용은 제외
        key = (self_law_name, norm_id)
        if key in seen:
            continue
        seen.add(key)
        refs.append({"law_name": self_law_name, "original_id_normalized": norm_id})

    return refs


## 6. 임베딩 생성

In [36]:
def embed_texts(texts: list[str], batch_size: int = 100) -> list[list[float]]:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch)
        all_embeddings.extend([d.embedding for d in response.data])
    return all_embeddings


## 7. Neo4j 스키마 준비 (제약조건 + 벡터 인덱스)

한 번만 실행하면 됩니다 (`IF NOT EXISTS`로 안전하게 재실행 가능).

In [37]:
def setup_schema():
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run("CREATE CONSTRAINT law_id IF NOT EXISTS FOR (l:LAW) REQUIRE l.id IS UNIQUE")
        session.run("CREATE CONSTRAINT article_id IF NOT EXISTS FOR (a:ARTICLE) REQUIRE a.id IS UNIQUE")

        session.run(f"""
        CREATE VECTOR INDEX article_vector_index IF NOT EXISTS
        FOR (a:ARTICLE) ON (a.embedding)
        OPTIONS {{
          indexConfig: {{
            `vector.dimensions`: {EMBED_DIM},
            `vector.similarity_function`: 'cosine'
          }}
        }}
        """)
    print("스키마(제약조건 + 벡터 인덱스) 준비 완료")


setup_schema()


스키마(제약조건 + 벡터 인덱스) 준비 완료


## 8. 단일 PDF 처리 (파싱 → 임베딩 → 적재 준비)

PDF 하나를 받아서 LAW 메타데이터 + 조문 리스트(임베딩, 인용 정보 포함)를 만듭니다.
Neo4j 적재는 다음 단계에서 모든 PDF를 처리한 뒤 한 번에 수행합니다 (인용관계가 다른 파일의 조문을 참조할 수도 있기 때문).

In [38]:
def process_pdf(pdf_path: str) -> dict:
    full_text = extract_full_text(pdf_path)
    meta = extract_law_metadata(pdf_path, full_text)
    law_id = meta["name"]  # 법령명을 LAW의 고유 id로 사용

    raw_articles = parse_articles(full_text)

    for art in raw_articles:
        art["references"] = extract_references(
            art["content"], art["original_id_normalized"], law_id
        )
        art["id"] = f"{law_id}::{art['original_id']}"
        art["law_id"] = law_id
        art["source_pdf"] = Path(pdf_path).name

    print(f"[{law_id}] 조문 {len(raw_articles)}개 파싱 완료 → 임베딩 생성 중...")
    contents = [a["content"] for a in raw_articles]
    if contents:
        embeddings = embed_texts(contents)
        for art, emb in zip(raw_articles, embeddings):
            art["embedding"] = emb

    return {
        "law": {"id": law_id, **meta},
        "articles": raw_articles,
    }


In [39]:
# total_refs가 0이라면, 실제 조문 본문에 "제OO조" 같은 텍스트가
# 어떤 형태로 들어있는지 직접 확인 (줄바꿈/공백/전각문자 등으로 정규식이 깨지는 경우가 흔함)
sample_article = sample_law_data["articles"][0]
print("샘플 조문 원문 (앞 500자):")
print(repr(sample_article["content"][:500]))

# 원문에 "조"라는 글자가 있는지, 그 앞뒤가 정규식과 어떻게 다른지 확인
import re
raw_matches = re.findall(r'.{0,5}조.{0,5}', sample_article["content"])
print("\n'조' 주변 텍스트 샘플:")
for m in raw_matches[:15]:
    print(" ", repr(m))


샘플 조문 원문 (앞 500자):
'이 훈령은「군인의 지위 및 복무에 관한 기본법」등에 따라 규정과 제도에 의한 부대관리와 운영체계 및\n지휘체계를 확립하고 전투임무에 전념할 수 있는 여건을 조성하는 데 기여함으로써 "강한 전사, 강한 군대"의 기\n풍을 조성하는데 목적이 있다.'

'조' 주변 텍스트 샘플:
  ' 여건을 조성하는 데'
  '풍을 조성하는데 '


## 9. Neo4j 적재

모든 PDF를 처리한 결과를 받아서:
1. LAW 노드 MERGE
2. ARTICLE 노드 MERGE (embedding 포함) + LAW-[:CONTAINS]->ARTICLE
3. 모든 조문이 적재된 후, REFERENCES 관계를 조번호 매칭으로 생성 (대상이 없으면 조용히 스킵)

In [40]:
def load_law_and_articles(session, law_data: dict):
    law = law_data["law"]
    session.run("""
    MERGE (l:LAW {id: $id})
    SET l.name = $name,
        l.law_type = $law_type,
        l.promulgation_date = $promulgation_date,
        l.effective_date = $effective_date
    """, **law)

    for art in law_data["articles"]:
        session.run("""
        MATCH (l:LAW {id: $law_id})
        MERGE (a:ARTICLE {id: $id})
        SET a.law_id = $law_id,
            a.name = $name,
            a.description = $content,
            a.original_id = $original_id,
            a.original_id_normalized = $original_id_normalized,
            a.source_pdf = $source_pdf,
            a.embedding = $embedding
        MERGE (l)-[:CONTAINS]->(a)
        """, **{k: art[k] for k in [
            "id", "law_id", "name", "content", "original_id",
            "original_id_normalized", "source_pdf", "embedding"
        ]})


def load_references(session, all_law_data: list[dict], verbose: bool = True):
    """인용관계를 연결한다. 디버깅을 위해 시도/성공/실패 건수와
    실패한 인용 대상 샘플을 같이 출력한다."""
    attempted = 0
    matched = 0
    unmatched_samples = []

    for law_data in all_law_data:
        for art in law_data["articles"]:
            for ref in art["references"]:
                attempted += 1
                result = session.run("""
                MATCH (src:ARTICLE {id: $src_id})
                MATCH (tgt:ARTICLE {law_id: $tgt_law_id, original_id_normalized: $tgt_norm_id})
                MERGE (src)-[:REFERENCES]->(tgt)
                RETURN count(*) AS c
                """,
                    src_id=art["id"],
                    tgt_law_id=ref["law_name"],
                    tgt_norm_id=ref["original_id_normalized"],
                )
                c = result.single()["c"]
                matched += c
                if c == 0 and len(unmatched_samples) < 20:
                    unmatched_samples.append({
                        "from": art["id"],
                        "target_law": ref["law_name"],
                        "target_norm_id": ref["original_id_normalized"],
                    })

    print(f"인용 시도: {attempted}건 / 실제 연결: {matched}건 / 매칭 실패: {attempted - matched}건")

    if verbose and unmatched_samples:
        print("\n매칭 실패 샘플 (최대 20개):")
        for s in unmatched_samples:
            print(f'  {s["from"]}  ->  law="{s["target_law"]}" norm_id="{s["target_norm_id"]}" (해당 조문을 DB에서 못 찾음)')

    return {"attempted": attempted, "matched": matched}


## 10. 전체 파이프라인 실행 (여러 PDF)

In [42]:
def run_pipeline(pdf_paths: list[str]):
    all_law_data = []
    for pdf_path in tqdm(pdf_paths, desc="PDF 처리 중"):
        law_data = process_pdf(pdf_path)
        all_law_data.append(law_data)

    print("\nNeo4j에 적재 중...")
    with driver.session(database=NEO4J_DATABASE) as session:
        for law_data in all_law_data:
            load_law_and_articles(session, law_data)
        # 모든 조문이 들어간 뒤에 인용관계 연결 (파일 간 상호 참조 지원)
        load_references(session, all_law_data)

    total_articles = sum(len(ld["articles"]) for ld in all_law_data)
    print(f"\n완료: 법령 {len(all_law_data)}개 / 조문 {total_articles}개 적재")
    return all_law_data


all_law_data = run_pipeline(pdf_paths)

PDF 처리 중:   0%|          | 0/16 [00:00<?, ?it/s]

[공무원보수규정] 조문 96개 파싱 완료 → 임베딩 생성 중...


PDF 처리 중:   6%|▋         | 1/16 [00:03<00:58,  3.90s/it]

[공무원수당 등에 관한 규정] 조문 46개 파싱 완료 → 임베딩 생성 중...


PDF 처리 중:  12%|█▎        | 2/16 [00:06<00:41,  2.97s/it]

...

[부대관리훈령] 조문 412개 파싱 완료 → 임베딩 생성 중...


PDF 처리 중:  94%|█████████▍| 15/16 [00:51<00:05,  5.25s/it]

[제대군인지원에 관한 법률] 조문 56개 파싱 완료 → 임베딩 생성 중...


PDF 처리 중: 100%|██████████| 16/16 [00:53<00:00,  3.32s/it]



Neo4j에 적재 중...
인용 시도: 2782건 / 실제 연결: 1931건 / 매칭 실패: 851건

매칭 실패 샘플 (최대 20개):
  공무원보수규정::제3조  ->  law="공공기관의 운영에 관한 법률" norm_id="4" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제4조의2  ->  law="공무원 수당 등에 관한 규정" norm_id="7-2" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제4조의2  ->  law="공무원 수당 등에 관한 규정" norm_id="23" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제6조  ->  law="공무원임용령" norm_id="29" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="국가공무원법" norm_id="26-2" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지방공무원법" norm_id="25-3" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="공무원임용령" norm_id="3-2" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지방공무원 임용령" norm_id="3-2" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="공무원임용령" norm_id="3-3" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지
방공무원 임용령" norm_id="3-5" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="국가공무원법" norm_id="71" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지방공무원법" norm_id="63" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="공무원임용령" norm_id="57-3" (해당 조문을 DB에서 못 찾음)
  공무원보수규정::제8조  ->  law="지방공무

## 11. 검증

적재가 잘 됐는지 확인하는 몇 가지 쿼리입니다.

In [45]:
def verify():
    with driver.session(database=NEO4J_DATABASE) as session:
        law_count = session.run("MATCH (l:LAW) RETURN count(l) AS c").single()["c"]
        article_count = session.run("MATCH (a:ARTICLE) RETURN count(a) AS c").single()["c"]
        ref_count = session.run("MATCH ()-[r:REFERENCES]->() RETURN count(r) AS c").single()["c"]
        print(f"LAW 노드: {law_count}개")
        print(f"ARTICLE 노드: {article_count}개")
        print(f"REFERENCES 관계: {ref_count}개")

        print("\n샘플 조문 5개:")
        results = session.run("MATCH (a:ARTICLE) RETURN a.id AS id, a.original_id AS oid LIMIT 5")
        for r in results:
            print(" -", r["id"])


verify()


DriverError: Driver closed

## 12. 정리

노트북 실행이 끝나면 driver를 닫아주세요.

In [44]:
driver.close()
